# Databricks: Bidirectional Ossie Sync

Keeps `sales_metric_view` and the shared Ossie file on S3 in agreement, in both
directions. A measure added here is published for Snowflake to pick up; a metric added in
Snowflake is applied here.

Runs two ways, and they call the same function:

- `run_once()` in this notebook, for when you do not want to wait
- a Databricks Job on a 1-minute schedule, for the background heartbeat

Why it does not ping-pong between the platforms: every decision compares a fingerprint of
what the model *means*, not when a file was written. Once the two sides agree, neither
writes. Comparing timestamps cannot work, because any write makes the writer the most
recent change.

Runbook: `docs/DEMO_RUNBOOK_BIDIRECTIONAL.md`

## Step 1 - Install the Apache Ossie Databricks converter

Snowflake reads and writes Ossie natively. Databricks does not yet, so this side
uses the open-source converter, pinned to a known commit.

For a tight demo loop, install `ossie-databricks` as a cluster library and delete
these two cells; on serverless the install costs most of a minute.

In [ ]:
%pip install "git+https://github.com/apache/ossie.git@01058aa416423cf43a74e7f9fb7f5f70981a418e#subdirectory=converters/databricks"

In [ ]:
dbutils.library.restartPython()

## Step 2 - Configuration

Catalog and schema match the Snowflake database and schema so the table references
inside the Ossie file resolve on both platforms.

In [ ]:
CATALOG       = "demos"
SCHEMA        = "ext_semantic_interop"
METRIC_VIEW   = f"{CATALOG}.{SCHEMA}.sales_metric_view"
MODEL_NAME    = "SALES_SV"           # keep the Snowflake model name across the round trip

# Ossie embeds Snowflake's 3-part table names; swap namespaces on the way through.
SF_NAMESPACE  = "DEMOS.EXT_SEMANTIC_INTEROP"
DBX_NAMESPACE = f"{CATALOG}.{SCHEMA}"

S3_BUCKET     = "s3://snowflake-ossie-interop"        # <-- your bucket
OSSIE_MODEL   = f"{S3_BUCKET}/ossie/sales_model.yaml"        # shared, either side writes
STATE_FILE    = f"{S3_BUCKET}/ossie/_state/databricks.json"  # this side only

PLATFORM        = "databricks"
ALLOWED         = BIDIRECTIONAL          # this side may both import and export
CONFLICT_WINNER = "snowflake"        # demoware: see docs/PRODUCTION_ARCHITECTURE.md

print(f"Metric View : {METRIC_VIEW}")
print(f"Shared model: {OSSIE_MODEL}")
print(f"Allowed     : {', '.join(ALLOWED)}")

## Step 3 - Shared sync logic

Generated from `assets/ossie_sync/` by `assets/build_notebooks.py`. Both platforms
carry an identical copy, which is the point: if the Snowflake and Databricks
fingerprints disagreed by a single character, the two sides would never converge
and the sync would write on every tick forever.

Edit `assets/ossie_sync/*.py` and re-run the build rather than editing this cell.

In [ ]:
import hashlib
import json
import re
from datetime import datetime, timezone

import yaml

from ossie_databricks import convert_metric_view_to_ossie, convert_ossie_to_metric_view


# --- BEGIN GENERATED: ossie_sync.fingerprint ---
# Generated from assets/ossie_sync/fingerprint.py by assets/build_notebooks.py.
# Edit that file and re-run the build; changes made here are overwritten.
"""Canonical semantic fingerprint for an Ossie document.

Why this exists
---------------
The Snowflake <-> Databricks round trip is not byte-stable. The same semantic model
comes back with different dataset name casing, `primary_key` renamed to `unique_keys`,
facts dropped, dialect labels rewritten, and expressions gaining or losing table
qualifiers (`SUM(orders.order_amount)` on one side, `SUM(order_amount)` on the other).

So a sync that compares raw YAML, or a hash of it, never sees the two sides as equal and
writes forever. A sync that compares file timestamps is worse: every write makes the
writer the most recent change, so the model ping-pongs between platforms.

`semantic_fingerprint` solves this by hashing only the part of the model that both
platforms can express, in a normalized form that survives the trip. Two models with the
same fingerprint are treated as the same model, which is what lets the sync go quiet.

What is included
----------------
    tables          alias and source table, lowercased, last path component only
    relationships   from, to, and the join columns
    dimensions      qualified dimension names, sorted
    metrics         name and expression, sorted, table qualifiers stripped

What is excluded, and why
-------------------------
    Ossie `version`         differs by spec revision (0.1.1 against 0.2.0.dev0)
    dialect labels          SNOWFLAKE against ANSI_SQL against DATABRICKS
    comments, descriptions  Databricks does not round-trip them
    FACTS                   Snowflake-only concept, dropped by the converter
    relationship names      the converter rewrites their casing
    primary keys            Snowflake `primary_key` becomes `unique_keys` and back
    field and key order     not semantically meaningful

Excluding these has a real cost: editing only a comment, or only a fact, propagates
nothing. That is the deliberate trade. Including them would mean the two sides never
agree and the sync would write on every tick forever.
"""



FINGERPRINT_VERSION = "1"

# Matches a leading table qualifier on a column reference, e.g. the "orders." in
# "orders.order_amount". Stripped so that SUM(orders.order_amount) on the Snowflake side
# and SUM(order_amount) on the Databricks side produce the same fingerprint.
_QUALIFIER = re.compile(r"\b[A-Za-z_]\w*\.(?=[A-Za-z_]\w*)")
_WHITESPACE = re.compile(r"\s+")


def normalize_expression(expr):
    """Reduce a SQL expression to a comparable form.

    Lowercases, collapses whitespace, strips table qualifiers, and removes spaces
    around punctuation so that formatting differences do not register as changes.

        >>> normalize_expression("SUM( orders.order_amount )")
        'sum(order_amount)'
        >>> normalize_expression("sum(order_amount)")
        'sum(order_amount)'
    """
    if not expr:
        return ""
    text = _QUALIFIER.sub("", str(expr))
    text = _WHITESPACE.sub(" ", text).strip().lower()
    for token in ("(", ")", ",", "+", "-", "*", "/"):
        text = text.replace(" " + token, token).replace(token + " ", token)
    return text


def _last_identifier(source):
    """DEMOS.EXT_SEMANTIC_INTEROP.ORDERS -> orders"""
    return str(source or "").split(".")[-1].strip().strip('"').lower()


def _pick_expression(expression_obj):
    """Return the first expression string from an Ossie expression object.

    Dialect is ignored on purpose. The same expression labelled SNOWFLAKE, ANSI_SQL or
    DATABRICKS is the same expression for fingerprint purposes.
    """
    if not isinstance(expression_obj, dict):
        return ""
    for dialect in expression_obj.get("dialects") or []:
        if dialect.get("expression"):
            return dialect["expression"]
    return ""


def semantic_projection(ossie_yaml):
    """Reduce an Ossie document to the platform-neutral structure that gets hashed.

    Returned separately from the hash so notebooks can print it and show exactly what
    is being compared. When a sync will not converge, diffing two projections is the
    fastest way to find out which field is to blame.
    """
    root = yaml.safe_load(ossie_yaml) if isinstance(ossie_yaml, str) else ossie_yaml
    models = root.get("semantic_model") or []

    tables, relationships, dimensions, metrics = [], [], [], []

    for model in models:
        datasets = model.get("datasets") or []

        for dataset in datasets:
            alias = _last_identifier(dataset.get("name"))
            tables.append({"alias": alias, "source": _last_identifier(dataset.get("source"))})

            for field in dataset.get("fields") or []:
                # Only dimensions are portable. Snowflake facts have no `dimension` key
                # and are dropped by the Databricks converter, so including them here
                # would break convergence.
                if "dimension" not in field:
                    continue
                dimensions.append(f"{alias}.{str(field.get('name','')).lower()}")

        for rel in model.get("relationships") or []:
            # Ossie spells the join columns from_columns / to_columns. Only the join
            # columns are fingerprinted; the relationship's own name is not, because the
            # converter rewrites its casing (ORDERS_TO_CUSTOMERS -> ORDERS_to_CUSTOMERS).
            relationships.append({
                "from": _last_identifier(rel.get("from")),
                "to": _last_identifier(rel.get("to")),
                "from_columns": sorted(_last_identifier(c) for c in rel.get("from_columns") or []),
                "to_columns": sorted(_last_identifier(c) for c in rel.get("to_columns") or []),
            })

        # Snowflake stores metrics inside datasets[*].custom_extensions as a JSON blob;
        # the Apache converter uses a top-level `metrics` list. Read both.
        for metric in model.get("metrics") or []:
            metrics.append({
                "name": str(metric.get("name", "")).lower(),
                "expr": normalize_expression(_pick_expression(metric.get("expression"))),
            })

        for dataset in datasets:
            for ext in dataset.get("custom_extensions") or []:
                if ext.get("vendor_name") != "SNOWFLAKE":
                    continue
                try:
                    blob = json.loads(ext.get("data") or "{}")
                except (ValueError, TypeError):
                    continue
                for metric in blob.get("metrics") or []:
                    metrics.append({
                        "name": str(metric.get("name", "")).lower(),
                        "expr": normalize_expression(metric.get("expr")),
                    })

    def dedupe(rows, key):
        seen, out = set(), []
        for row in rows:
            marker = key(row)
            if marker not in seen:
                seen.add(marker)
                out.append(row)
        return out

    return {
        "fingerprint_version": FINGERPRINT_VERSION,
        "tables": sorted(dedupe(tables, lambda t: t["alias"]), key=lambda t: t["alias"]),
        "relationships": sorted(
            dedupe(relationships, lambda r: (r["from"], r["to"], tuple(r["from_columns"]))),
            key=lambda r: (r["from"], r["to"]),
        ),
        "dimensions": sorted(set(dimensions)),
        "metrics": sorted(dedupe(metrics, lambda m: m["name"]), key=lambda m: m["name"]),
    }


def semantic_fingerprint(ossie_yaml):
    """sha256 over the canonical projection. Stable across the round trip."""
    canonical = json.dumps(semantic_projection(ossie_yaml), sort_keys=True, separators=(",", ":"))
    return "sha256:" + hashlib.sha256(canonical.encode("utf-8")).hexdigest()


def describe(fingerprint):
    """Short form for log lines and notebook output."""
    if not fingerprint:
        return "(none)"
    return fingerprint.split(":")[-1][:12]
# --- END GENERATED: ossie_sync.fingerprint ---

# --- BEGIN GENERATED: ossie_sync.decide ---
# Generated from assets/ossie_sync/decide.py by assets/build_notebooks.py.
# Edit that file and re-run the build; changes made here are overwritten.
"""The sync decision: compare three fingerprints, return one verdict.

Both platforms and both architectures run this same function. The only thing that varies
is `allowed`, which is what stops the unidirectional variant from being a fork of the
bidirectional one.

The three inputs
----------------
    local   fingerprint of the model as it exists on this platform right now
    remote  fingerprint of the shared Ossie file on S3
    base    fingerprint this platform last agreed on, from its own state file

`base` is what makes this terminate. Without it there is no way to tell "the other side
changed" from "I changed", so both sides write and the model ping-pongs forever. With it,
each side can see which of the two moved, act once, record the new base, and go quiet.
"""

NO_CHANGE = "NO_CHANGE"
ADOPT = "ADOPT"
IMPORT = "IMPORT"
EXPORT = "EXPORT"
CONFLICT = "CONFLICT"
REVERT_LOCAL_DRIFT = "REVERT_LOCAL_DRIFT"

BIDIRECTIONAL = ("IMPORT", "EXPORT")
SNOWFLAKE_MANAGED_SOURCE = ("EXPORT",)      # Snowflake in the managed architecture
SNOWFLAKE_MANAGED_MIRROR = ("IMPORT",)      # Databricks in the managed architecture

REASONS = {
    NO_CHANGE: "local and shared model agree, nothing to do",
    ADOPT: "no recorded base, taking the shared model as the starting point",
    IMPORT: "shared model changed, replacing the local model",
    EXPORT: "local model changed, publishing to the shared Ossie file",
    CONFLICT: "both sides changed since the last agreement",
    REVERT_LOCAL_DRIFT: "local edit is not authoritative, restoring from the shared model",
}


class Decision:
    """A verdict plus the fingerprints that produced it, so it can be logged and read."""

    def __init__(self, action, reason, local, remote, base):
        self.action = action
        self.reason = reason
        self.local = local
        self.remote = remote
        self.base = base

    @property
    def writes(self):
        return self.action in (IMPORT, EXPORT, ADOPT, REVERT_LOCAL_DRIFT)

    def __str__(self):
        return f"{self.action} - {self.reason}"

    def to_dict(self):
        return {
            "action": self.action,
            "reason": self.reason,
            "local_fingerprint": self.local,
            "remote_fingerprint": self.remote,
            "base_fingerprint": self.base,
        }


def decide(local, remote, base, allowed=BIDIRECTIONAL, conflict_winner=None, platform=None):
    """Return a Decision.

    allowed
        Which write directions this platform may take. Bidirectional passes both.
        The managed architecture passes ("EXPORT",) on Snowflake and ("IMPORT",) on
        Databricks; an EXPORT that is not allowed becomes REVERT_LOCAL_DRIFT.

    conflict_winner, platform
        When both sides changed, the platform named by `conflict_winner` keeps its
        version. Anything else imports. Demoware: the losing edit is discarded with
        nothing more than a log line. See docs/PRODUCTION_ARCHITECTURE.md.
    """
    def verdict(action):
        return Decision(action, REASONS[action], local, remote, base)

    if local and remote and local == remote:
        return verdict(NO_CHANGE)

    if not local:
        # Nothing here yet, so there is no local change to protect.
        return verdict(ADOPT if remote else NO_CHANGE)

    if not remote:
        # Local model exists but the shared file does not.
        return verdict(EXPORT if EXPORT in allowed else NO_CHANGE)

    if base is None:
        return verdict(ADOPT)

    if local == base:
        action = IMPORT
    elif remote == base:
        action = EXPORT
    else:
        if conflict_winner and platform and conflict_winner == platform:
            return verdict(EXPORT if EXPORT in allowed else CONFLICT)
        if conflict_winner and platform:
            return verdict(IMPORT if IMPORT in allowed else CONFLICT)
        return verdict(CONFLICT)

    if action == EXPORT and EXPORT not in allowed:
        # Managed architecture: a locally edited mirror is drift, not a contribution.
        return verdict(REVERT_LOCAL_DRIFT)
    if action == IMPORT and IMPORT not in allowed:
        return verdict(NO_CHANGE)

    return verdict(action)


def next_base(decision):
    """The fingerprint to record after acting, or None to leave the base unchanged."""
    if decision.action in (IMPORT, ADOPT, REVERT_LOCAL_DRIFT):
        return decision.remote
    if decision.action == EXPORT:
        return decision.local
    return None
# --- END GENERATED: ossie_sync.decide ---

# --- BEGIN GENERATED: ossie_sync.state ---
# Generated from assets/ossie_sync/state.py by assets/build_notebooks.py.
# Edit that file and re-run the build; changes made here are overwritten.
"""Per-platform sync state, stored as JSON next to the shared Ossie file.

    s3://<bucket>/ossie/
      sales_model.yaml            shared model, either side may write
      _state/snowflake.json       written only by Snowflake
      _state/databricks.json      written only by Databricks

One writer per file, so there is no lock and no race. Each side reads only its own state
to answer "what did I last agree to", which is the `base` argument to decide().

Reading and writing the file is left to the caller, because the two runtimes do it very
differently: Databricks has dbutils.fs, Snowflake has stage COPY INTO. These helpers only
handle the JSON shape.
"""


STATE_VERSION = "1"


def new_state(base_fingerprint=None, last_action=None, platform=None):
    return {
        "state_version": STATE_VERSION,
        "base_fingerprint": base_fingerprint,
        "last_action": last_action,
        "by": platform,
        "at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    }


def parse_state(text):
    """Tolerant read. A missing, empty or corrupt state file means no recorded base.

    Returning an empty state rather than raising is deliberate: decide() treats a base of
    None as ADOPT, which is the safe first-run behaviour and also the recovery path if
    someone deletes the file mid-demo.
    """
    if not text:
        return new_state()
    try:
        loaded = json.loads(text)
    except (ValueError, TypeError):
        return new_state()
    if not isinstance(loaded, dict):
        return new_state()
    return {
        "state_version": loaded.get("state_version", STATE_VERSION),
        "base_fingerprint": loaded.get("base_fingerprint"),
        "last_action": loaded.get("last_action"),
        "by": loaded.get("by"),
        "at": loaded.get("at"),
    }


def base_of(text):
    """The recorded base fingerprint from raw state-file text, or None."""
    return parse_state(text).get("base_fingerprint")


def serialize_state(state):
    return json.dumps(state, indent=2, sort_keys=True)


def state_after(decision, platform, next_base_fingerprint):
    """Build the state to persist after acting on a decision."""
    return new_state(
        base_fingerprint=next_base_fingerprint or decision.base,
        last_action=decision.action,
        platform=platform,
    )
# --- END GENERATED: ossie_sync.state ---

# --- BEGIN GENERATED: ossie_sync.shim ---
# Generated from assets/ossie_sync/shim.py by assets/build_notebooks.py.
# Edit that file and re-run the build; changes made here are overwritten.
# Licensed under Apache-2.0 (this file is original to the demo, not from apache/ossie).
"""Bridge between Snowflake's Ossie dialect and the Apache Ossie Databricks converter.

Why this exists
---------------
Snowflake's SYSTEM$READ_OSSIE_YAML_FROM_SEMANTIC_VIEW emits Ossie **0.1.1** and the
vendored Apache converter tracks **0.2.0.dev0** (an exact-match check). The two spec
revisions differ in three concrete ways that this module reconciles:

1. version string           0.1.1                 <->  0.2.0.dev0
2. expression dialect       SNOWFLAKE             <->  ANSI_SQL / DATABRICKS
3. metric placement         dataset custom_ext    <->  model-level `metrics`

Item (3) is the load-bearing one: Snowflake stores metrics inside
`datasets[*].custom_extensions[SNOWFLAKE].data` as a JSON blob, while the Apache
converter reads a top-level `metrics` list. Without hoisting, the generated Metric
View has no measures at all.

The transforms are deliberately narrow and reversible so the interop story stays
honest: the semantic content (names, expressions, relationships) is untouched; only
the envelope (version tag, dialect label, metric location) is adapted.
"""



CONVERTER_OSSIE_VERSION = "0.2.0.dev0"   # what the vendored Apache converter requires
SNOWFLAKE_OSSIE_VERSION = "0.1.1"        # what Snowflake emits / expects on import
SNOWFLAKE_DIALECT = "SNOWFLAKE"
ANSI_DIALECT = "ANSI_SQL"
DATABRICKS_DIALECT = "DATABRICKS"


def _relabel_dialects(expression_obj, frm, to):
    """Relabel every `dialect: <frm>` to `<to>` inside an Ossie expression object."""
    if not isinstance(expression_obj, dict):
        return
    for d in expression_obj.get("dialects", []) or []:
        if d.get("dialect") == frm:
            d["dialect"] = to


def snowflake_to_converter(ossie_yaml, drop_fact_fields=True):
    """Snowflake Ossie 0.1.1  ->  Apache-converter-ready Ossie 0.2.0.dev0.

    - bumps the version string
    - relabels SNOWFLAKE-dialect expressions to ANSI_SQL (the converter only reads
      DATABRICKS/ANSI_SQL)
    - hoists metrics out of each dataset's SNOWFLAKE custom_extension up to a
      model-level `metrics` list, stripping the `<dataset>.` qualifier so measure
      expressions are bare fact columns (the Databricks idiom: SUM(order_amount))
    - by default drops fact fields (those with no `dimension` marker) so they do not
      become groupable Metric View dimensions; they live on inside measure expressions
    """
    root = yaml.safe_load(ossie_yaml)
    root["version"] = CONVERTER_OSSIE_VERSION

    for model in root.get("semantic_model", []) or []:
        hoisted = []
        for ds in model.get("datasets", []) or []:
            ds_name = ds.get("name", "")
            qual = re.compile(re.escape(ds_name) + r"\.", re.IGNORECASE)

            # Hoist metrics from this dataset's SNOWFLAKE custom_extension.
            kept_ext = []
            for ext in ds.get("custom_extensions", []) or []:
                if ext.get("vendor_name") == SNOWFLAKE_DIALECT:
                    blob = json.loads(ext.get("data") or "{}")
                    for m in blob.get("metrics", []) or []:
                        expr = qual.sub("", m["expr"])  # SUM(orders.order_amount) -> SUM(order_amount)
                        hoisted.append({
                            "name": m["name"],
                            "expression": {
                                "dialects": [{"dialect": ANSI_DIALECT, "expression": expr}]
                            },
                        })
                else:
                    kept_ext.append(ext)
            if kept_ext:
                ds["custom_extensions"] = kept_ext
            else:
                ds.pop("custom_extensions", None)

            # Fields: relabel dialects, strip field-level SNOWFLAKE extensions, and
            # optionally drop facts (kept only if they carry a `dimension` marker).
            new_fields = []
            for f in ds.get("fields", []) or []:
                _relabel_dialects(f.get("expression"), SNOWFLAKE_DIALECT, ANSI_DIALECT)
                f.pop("custom_extensions", None)
                if drop_fact_fields and "dimension" not in f:
                    continue
                new_fields.append(f)
            if new_fields:
                ds["fields"] = new_fields
            else:
                ds.pop("fields", None)

        if hoisted:
            model["metrics"] = (model.get("metrics", []) or []) + hoisted

        # Metrics that are ALREADY model-level need the same treatment as hoisted ones.
        # This is the second-hop case: converter_to_snowflake leaves metrics at model
        # level, labelled SNOWFLAKE and re-qualified as SUM(ORDERS.order_amount). Without
        # this the converter finds "no DATABRICKS/ANSI_SQL dialect" and silently drops
        # every metric, so the second round trip loses the whole measure set and the sync
        # never converges.
        ds_names = [ds.get("name", "") for ds in model.get("datasets", []) or [] if ds.get("name")]
        for m in model.get("metrics", []) or []:
            _relabel_dialects(m.get("expression"), SNOWFLAKE_DIALECT, ANSI_DIALECT)
            for d in (m.get("expression") or {}).get("dialects", []) or []:
                if "expression" not in d:
                    continue
                for ds_name in ds_names:
                    d["expression"] = re.sub(
                        re.escape(ds_name) + r"\.", "", d["expression"], flags=re.IGNORECASE
                    )

    return yaml.safe_dump(root, sort_keys=False)


def converter_to_snowflake(ossie_yaml, dialect=SNOWFLAKE_DIALECT, model_name=None):
    """Apache-converter Ossie 0.2.0.dev0  ->  Snowflake-importable Ossie 0.1.1.

    Databricks Metric Views don't carry Snowflake's fact/dimension distinction, and the
    forward trip dropped the fact columns, so this reverse trip must rebuild what
    Snowflake's importer needs:

    - resets the version string and relabels DATABRICKS-dialect expressions to SNOWFLAKE
    - qualifies each measure's bare columns with the fact table (COUNT(order_id) ->
      COUNT(ORDERS.order_id)); Snowflake derived metrics need a logical-table-qualified
      column
    - reconstructs the referenced fact columns as fields on the fact dataset (no
      `dimension` marker => facts) so the metric expressions resolve
    - marks every field on a non-fact (joined) dataset with `dimension: {}` so Snowflake
      classifies region/customer_name as dimensions, not facts
    - optionally renames the model (the new semantic view name) without touching the
      fact dataset name
    """
    root = yaml.safe_load(ossie_yaml)
    root["version"] = SNOWFLAKE_OSSIE_VERSION
    for model in root.get("semantic_model", []) or []:
        datasets = model.get("datasets", []) or []

        # Normalize dataset names to uppercase. Snowflake's importer resolves
        # metric table-qualifiers against dataset names case-sensitively, and
        # unquoted Snowflake identifiers are uppercase internally. Without this,
        # a lowercase dataset name (from a Databricks table named "orders")
        # causes "invalid identifier" errors on import.
        name_map = {}
        for ds in datasets:
            old_name = ds["name"]
            ds["name"] = old_name.upper()
            if old_name != ds["name"]:
                name_map[old_name] = ds["name"]
        for rel in model.get("relationships", []) or []:
            if "from" in rel:
                rel["from"] = rel["from"].upper()
            if "to" in rel:
                rel["to"] = rel["to"].upper()

        fact_ds_name = datasets[0]["name"] if datasets else None

        # Fields: relabel dialects; mark joined-dataset fields as dimensions.
        for ds in datasets:
            is_fact = ds.get("name") == fact_ds_name
            for f in ds.get("fields", []) or []:
                _relabel_dialects(f.get("expression"), DATABRICKS_DIALECT, dialect)
                if not is_fact:
                    f.setdefault("dimension", {})

        # Metrics: relabel, qualify bare columns with the fact table, then collect
        # every fact column the metric references - whether it arrived bare
        # (COUNT(order_id)) or already qualified (SUM(ORDERS.order_qty)) - so the
        # fact fields can be rebuilt for all of them.
        fact_cols = []
        ref_re = re.compile(re.escape(fact_ds_name) + r"\.([A-Za-z_]\w*)") if fact_ds_name else None
        for m in model.get("metrics", []) or []:
            _relabel_dialects(m.get("expression"), DATABRICKS_DIALECT, dialect)
            if not fact_ds_name:
                continue
            for d in (m.get("expression") or {}).get("dialects", []) or []:
                if "expression" in d:
                    # Fix case of pre-existing qualifiers that arrived lowercase
                    # (e.g. "orders.order_id" -> "ORDERS.order_id") so they match
                    # the uppercased dataset name.
                    for old, new in name_map.items():
                        d["expression"] = re.sub(
                            r"\b" + re.escape(old) + r"\.", new + ".", d["expression"])
                    d["expression"] = _qualify_columns(d["expression"], fact_ds_name)
                    for c in ref_re.findall(d["expression"]):
                        if c not in fact_cols:
                            fact_cols.append(c)

        # Rebuild the referenced columns as fact fields on the fact dataset.
        if fact_ds_name and fact_cols:
            fact_ds = datasets[0]
            existing = {f["name"].lower() for f in fact_ds.get("fields", []) or []}
            flds = fact_ds.setdefault("fields", [])
            for c in fact_cols:
                if c.lower() not in existing:
                    flds.append({
                        "name": c.upper(),
                        "expression": {"dialects": [{"dialect": dialect, "expression": c}]},
                    })

        if model_name:
            model["name"] = model_name
    return yaml.safe_dump(root, sort_keys=False)


# Prefix each bare column with the fact table; SQL function names (followed by "(")
# and already-qualified names (customer.c_name, ORDERS.order_qty) are left alone.
def _qualify_columns(expr, table):
    return re.sub(
        r"(?<![\w.])([A-Za-z_]\w*)(?!\s*\()(?![\w.])",
        lambda m: f"{table}.{m.group(1)}",
        expr,
    )
# --- END GENERATED: ossie_sync.shim ---

## Step 4 - Databricks-side helpers

Everything platform-specific lives here: reading and writing S3 through
`dbutils.fs`, reading and replacing the Metric View, and the two conversions.
The three fingerprints at the bottom are what the decision consumes.

In [ ]:
UNSUPPORTED_JOIN_FIELDS = ("rely",)   # older Databricks serdes reject these


# ---------------------------------------------------------------- S3 (dbutils.fs)

def read_s3(path):
    """Return the contents of an S3 object, or None if it does not exist."""
    try:
        return dbutils.fs.head(path, 1024 * 1024)
    except Exception:
        return None


def write_s3(path, body):
    dbutils.fs.put(path, body, overwrite=True)


def repair_backslashes(text):
    """Undo backslash doubling that some cross-platform transfers introduce."""
    if text is None:
        return None
    try:
        yaml.safe_load(text)
        return text
    except yaml.YAMLError:
        return text.replace("\\\\", "\\")


# ------------------------------------------------------- Metric View (read/write)

def metric_view_yaml():
    """The deployed Metric View's YAML body, or None if the view does not exist."""
    try:
        ddl = spark.sql(f"SHOW CREATE TABLE {METRIC_VIEW}").collect()[0][0]
    except Exception:
        return None
    start = ddl.index("$") + 2
    return ddl[start:ddl.index("$", start)].strip()


def overwrite_metric_view(mv_yaml):
    spark.sql(
        "CREATE OR REPLACE VIEW " + METRIC_VIEW
        + " WITH METRICS LANGUAGE YAML AS $$\n" + mv_yaml + "\n$$"
    )


def measures_of(mv_yaml):
    return [m["name"] for m in (yaml.safe_load(mv_yaml) or {}).get("measures", [])]


# ------------------------------------------------------------------ conversions

def strip_unsupported_fields(mv_yaml):
    mv = yaml.safe_load(mv_yaml)

    def clean(joins):
        for join in joins or []:
            for field in UNSUPPORTED_JOIN_FIELDS:
                join.pop(field, None)
            clean(join.get("joins"))

    clean(mv.get("joins"))
    return yaml.safe_dump(mv, sort_keys=False)


def ossie_to_metric_view(ossie_yaml):
    """Shared Ossie YAML -> Metric View YAML body."""
    mv_yaml = convert_ossie_to_metric_view(snowflake_to_converter(ossie_yaml))
    return strip_unsupported_fields(mv_yaml.replace(SF_NAMESPACE, DBX_NAMESPACE))


def metric_view_to_ossie(mv_yaml):
    """Metric View YAML body -> Snowflake-importable Ossie YAML."""
    ossie = convert_metric_view_to_ossie(mv_yaml).replace(DBX_NAMESPACE, SF_NAMESPACE)
    return converter_to_snowflake(ossie, model_name=MODEL_NAME)


# ------------------------------------------------- the three fingerprints

def local_fingerprint():
    """Fingerprint of the Metric View as it exists right now, or None if absent."""
    mv_yaml = metric_view_yaml()
    return semantic_fingerprint(metric_view_to_ossie(mv_yaml)) if mv_yaml else None


def remote_fingerprint(ossie_yaml):
    return semantic_fingerprint(ossie_yaml) if ossie_yaml else None


def base_fingerprint():
    return base_of(read_s3(STATE_FILE))


def save_base(decision, new_base):
    write_s3(STATE_FILE, serialize_state(state_after(decision, PLATFORM, new_base)))

## Step 5 - The sync itself

One function, two triggers. The scheduled job calls `run_once()` exactly as the
next cell does, so what you demonstrate by hand is what runs in the background.

In [ ]:
def run_once(verbose=True):
    """Compare, decide, act once. Same function the scheduled job runs.

    Returns a dict so the Jobs run history shows what happened without opening logs.
    """
    shared_ossie = repair_backslashes(read_s3(OSSIE_MODEL))

    local = local_fingerprint()
    remote = remote_fingerprint(shared_ossie)
    base = base_fingerprint()

    decision = decide(local, remote, base, allowed=ALLOWED,
                      conflict_winner=CONFLICT_WINNER, platform=PLATFORM)

    if verbose:
        print(f"Local  : {describe(local)}   (Metric View)")
        print(f"Remote : {describe(remote)}   (sales_model.yaml)")
        print(f"Base   : {describe(base)}   (last agreed)")
        print(f"Verdict: {decision}")

    result = decision.to_dict()

    if decision.action in (IMPORT, ADOPT, REVERT_LOCAL_DRIFT):
        mv_yaml = ossie_to_metric_view(shared_ossie)
        overwrite_metric_view(mv_yaml)
        result["measures"] = measures_of(mv_yaml)
        if verbose:
            print(f"Applied to {METRIC_VIEW}")
            print("Measures:", ", ".join(result["measures"]) or "(none)")

    elif decision.action == EXPORT:
        published = metric_view_to_ossie(metric_view_yaml())
        write_s3(OSSIE_MODEL, published)
        result["measures"] = measures_of(metric_view_yaml())
        if verbose:
            print(f"Published to {OSSIE_MODEL}")

    elif decision.action == CONFLICT and verbose:
        print("Both sides changed since the last agreement. Nothing written.")
        print("Resolve by re-running after one side is reconciled.")

    new_base = next_base(decision)
    if new_base:
        save_base(decision, new_base)

    return result


def reset_demo():
    """Return this side to a clean pre-demo state: no Metric View, no recorded base."""
    spark.sql(f"DROP VIEW IF EXISTS {METRIC_VIEW}")
    write_s3(STATE_FILE, serialize_state(new_state(platform=PLATFORM)))
    print(f"Dropped {METRIC_VIEW} and cleared {STATE_FILE}")
    print("The next run_once() will report ADOPT and build the view from the shared model.")

## Step 6 - Run it

Run `reset_demo()` first if you want to show the Metric View being created from
nothing. Otherwise go straight to `run_once()`.

In [ ]:
# reset_demo()   # uncomment to start the demo from a clean slate

In [ ]:
result = run_once()

Run it a second time. `NO_CHANGE` means the two sides agree and nothing was written,
which is the loop termination the whole design turns on.

In [ ]:
_ = run_once()

## Step 7 - Query the Metric View

In [ ]:
display(spark.sql(f"""
  SELECT region,
         MEASURE(order_count)        AS order_count,
         MEASURE(total_order_amount) AS total_order_amount
    FROM {METRIC_VIEW}
   GROUP BY region ORDER BY region
"""))
# expect EAST 5/750, WEST 5/700

## Step 8 - Add a measure on this side

This is the Databricks-authored change that travels back to Snowflake. After
running it, `run_once()` reports `EXPORT`.

In [ ]:
mv = yaml.safe_load(metric_view_yaml())
mv.setdefault("measures", []).append({"name": "TOTAL_QUANTITY", "expr": "SUM(order_qty)"})
overwrite_metric_view(yaml.safe_dump(mv, sort_keys=False))
print("Added TOTAL_QUANTITY:", ", ".join(measures_of(metric_view_yaml())))

In [ ]:
display(spark.sql(f"""
  SELECT region, MEASURE(total_quantity) AS total_quantity
    FROM {METRIC_VIEW}
   GROUP BY region ORDER BY region
"""))
# expect EAST 12, WEST 11

In [ ]:
result = run_once()   # expect EXPORT

## Step 9 - Report the outcome to the Jobs run history

In [ ]:
dbutils.notebook.exit(json.dumps(result))

## Scheduling this notebook

Workflows, Jobs, Create job:

- Task type Notebook, pointing at this notebook
- Schedule every minute, offset roughly 30 seconds from the Snowflake task so the two
  are not writing at the same instant
- Max concurrent runs 1, so a slow run never overlaps the next tick
- Serverless, or a small single-node cluster kept warm

The run output carries the verdict, so the run history reads as a column of `NO_CHANGE` with an
occasional `IMPORT` or `EXPORT` where something actually changed.

Pause the job when the demo ends.